In [68]:
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from preprocessing import load_data, sparse_to_tuple, preprocess_graph


from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix, to_undirected, remove_self_loops
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, VGAE

np.random.seed(42)
torch.manual_seed(42)
# ----------------------------
# Build Data
# ----------------------------

adj, features, labels = load_data('baron3', './data/baron3', True)


def build_pyg_data(adj, features, labels=None, make_undirected=True, remove_diag=True):
    adj = sp.coo_matrix(adj)
    n, m = adj.shape
    if n != m:
        raise ValueError(f"adj must be square, got {adj.shape}")

    if remove_diag:
        adj = adj - sp.diags(adj.diagonal(), offsets=0, shape=adj.shape, format="coo")
        adj.eliminate_zeros()

    edge_index, _ = from_scipy_sparse_matrix(adj)

    if make_undirected:
        edge_index = to_undirected(edge_index)

    if sp.issparse(features):
        x = torch.from_numpy(features.toarray()).float()
    else:
        x = torch.from_numpy(np.asarray(features)).float()

    data = Data(x=x, edge_index=edge_index)

    if labels is not None:
        data.y = torch.from_numpy(np.asarray(labels)).long()

    return data


# ----------------------------
# Encoder
# ----------------------------
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, latent_channels,activation=torch.relu):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv_mu = GCNConv(hidden_channels, latent_channels)
        self.conv_logvar = GCNConv(hidden_channels, latent_channels)
        self.activation = activation

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = self.activation(h)
        return self.conv_mu(h, edge_index), self.conv_logvar(h, edge_index)


# ----------------------------
# Version-robust accessors
# ----------------------------
def get_pos_neg_edges(split_data):
    """
    Return (pos_edge_index, neg_edge_index) each with shape [2, E].
    Handles common PyG attribute variants.
    """
    # Newer versions:
    if hasattr(split_data, "pos_edge_label_index") and hasattr(split_data, "neg_edge_label_index"):
        return split_data.pos_edge_label_index, split_data.neg_edge_label_index

    # Older versions:
    if hasattr(split_data, "pos_edge_index") and hasattr(split_data, "neg_edge_index"):
        return split_data.pos_edge_index, split_data.neg_edge_index

    # Unified label format:
    if hasattr(split_data, "edge_label_index") and hasattr(split_data, "edge_label"):
        idx = split_data.edge_label_index
        y = split_data.edge_label
        pos = idx[:, y == 1]
        neg = idx[:, y == 0]
        return pos, neg

    # If we reach here, print keys for debugging:
    keys = split_data.keys() if callable(getattr(split_data, "keys", None)) else []
    raise RuntimeError(f"Could not find pos/neg edges on split data. Available keys: {keys}")


# ----------------------------
# Train / Eval
# ----------------------------
def train_one_epoch(model, optimizer, train_data):
    model.train()
    optimizer.zero_grad()

    z = model.encode(train_data.x, train_data.edge_index)

    pos_edge_index, _ = get_pos_neg_edges(train_data)

    # Use only positives; VGAE.recon_loss will sample negatives internally.
    recon = model.recon_loss(z, pos_edge_index)
    kl = (1.0 / train_data.num_nodes) * model.kl_loss()

    loss = recon + kl
    loss.backward()
    optimizer.step()
    return float(loss), float(recon), float(kl)


@torch.no_grad()
def evaluate(model, split_data):
    model.eval()
    z = model.encode(split_data.x, split_data.edge_index)
    print(split_data)
    pos_edge_index, neg_edge_index = get_pos_neg_edges(split_data)
    auc, ap = model.test(z, pos_edge_index, neg_edge_index)
    return float(auc), float(ap)


# ----------------------------
# Runner
# ----------------------------
def run_vgae(
    adj,
    features,
    labels=None,
    make_undirected=True,
    hidden_channels=64,
    latent_channels=32,
    lr=0.01,
    weight_decay=0.0,
    epochs=200,
    val_ratio=0.05,
    test_ratio=0.10,
    neg_sampling_ratio=1.0,
    seed=42,
    device=None
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    data = build_pyg_data(adj, features, labels, make_undirected=make_undirected)

    # Remove self-loops before splitting
    ei, _ = remove_self_loops(data.edge_index)
    data.edge_index = ei

    splitter = RandomLinkSplit(
        num_val=val_ratio,
        num_test=test_ratio,
        is_undirected=make_undirected,
        add_negative_train_samples=True,
        neg_sampling_ratio=neg_sampling_ratio
    )

    train_data, val_data, test_data = splitter(data)

    # Sanity checks that will NOT crash due to naming differences
    _ = get_pos_neg_edges(train_data)
    _ = get_pos_neg_edges(val_data)
    _ = get_pos_neg_edges(test_data)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    train_data = train_data.to(device)
    val_data = val_data.to(device)
    test_data = test_data.to(device)
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = VGAE(GCNEncoder(train_data.num_features, hidden_channels, latent_channels)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_ap = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        loss, recon, kl = train_one_epoch(model, optimizer, train_data)
        val_auc, val_ap = evaluate(model, val_data)

        if val_ap > best_val_ap:
            best_val_ap = val_ap
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch == 1 or epoch % 20 == 0:
            print(
                f"Epoch {epoch:04d} | loss={loss:.4f} recon={recon:.4f} kl={kl:.4f} | "
                f"val_auc={val_auc:.4f} val_ap={val_ap:.4f}"
            )

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    test_auc, test_ap = evaluate(model, test_data)
    print(f"Best val AP={best_val_ap:.4f} | Test AUC={test_auc:.4f} Test AP={test_ap:.4f}")

    return model, (train_data, val_data, test_data)


# Example:
# model, splits = run_vgae(adj, features, labels, epochs=200, neg_sampling_ratio=5.0)

In [69]:
model, (train_data, val_data, test_data) = run_vgae(adj, features, labels, epochs=50)

Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
Epoch 0001 | loss=332084.2812 recon=32.8975 kl=332051.3750 | val_auc=0.5148 val_ap=0.5076
Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
Data(x=[3605, 1200], edge_index=[2, 22172], y=[3605], edge_label=[1304], edge_label_index=[2, 1304])
D